In [1]:
%load_ext autoreload
%autoreload 2

In [24]:
from pydrake.multibody.plant import AddMultibodyPlantSceneGraph
from pydrake.systems.framework import DiagramBuilder
from pydrake.multibody.parsing import Parser
import os
from pydrake.all import (
    LoadModelDirectives, ProcessModelDirectives, RevoluteJoint, 
    RationalForwardKinematics, CspaceFreePolytope, SeparatingPlaneOrder,
    RigidTransform, RotationMatrix, Rgba,
    AffineSubspace, MathematicalProgram, Solve,
    MeshcatVisualizer, StartMeshcat,
    PointCloud, RandomGenerator, SceneGraphCollisionChecker,
    RobotDiagramBuilder, MeshcatVisualizerParams,
    IrisZo, HPolyhedron, IrisZoOptions,
    AffineBall, Simulator
)
import numpy as np
# from pydrake.geometry.optimization_dev import (CspaceFreePolytope, SeparatingPlaneOrder)
from iris_plant_visualizer import IrisPlantVisualizer
from pydrake.geometry import Role
from pydrake.geometry.optimization import IrisOptions, HPolyhedron, Hyperellipsoid, LoadIrisRegionsYamlFile, SaveIrisRegionsYamlFile
from pydrake.solvers import MosekSolver, CommonSolverOption, SolverOptions, ScsSolver
import time
from pydrake.all import ModelVisualizer

In [3]:
def visualise_IRIS(regions, plant, plant_context, seed=42, num_sample=10000, colors=None):       
    world_frame = plant.world_frame()
    ee_frame = plant.GetFrameByName("iiwa_frame_ee")

    rng = RandomGenerator(seed)

    # Allow caller to input custom colors
    if colors is None:
        colors = [
                    Rgba(0.5,0.0,0.0,0.5),
                    Rgba(0.0,0.5,0.0,0.5),
                    Rgba(0.0,0.0,0.5,0.5),
                    Rgba(0.5,0.5,0.0,0.5),
                    Rgba(0.5,0.0,0.5,0.5),
                    Rgba(0.0,0.5,0.5,0.5),
                    Rgba(0.2,0.2,0.2,0.5),
                    Rgba(0.5,0.2,0.0,0.5),
                    Rgba(0.2,0.5,0.0,0.5),
                    Rgba(0.5,0.0,0.2,0.5),
                    Rgba(0.2,0.0,0.5,0.5),
                    Rgba(0.0,0.5,0.2,0.5),
                    Rgba(0.0,0.2,0.5,0.5),
                ]

    for i in range(len(regions)):
        region = regions[i]

        xyzs = []  # List to hold XYZ positions of configurations in the IRIS region

        q_sample = region.UniformSample(rng)

        plant.SetPositions(plant_context, q_sample)
        xyzs.append(plant.CalcRelativeTransform(plant_context, frame_A=world_frame, frame_B=ee_frame).translation())

        for _ in range(num_sample-1):
            prev_sample = q_sample
            q_sample = region.UniformSample(rng, prev_sample)

            plant.SetPositions(plant_context, q_sample)
            xyzs.append(plant.CalcRelativeTransform(plant_context, frame_A=world_frame, frame_B=ee_frame).translation())

        # Create pointcloud from sampled point in IRIS region in order to plot in Meshcat
        xyzs = np.array(xyzs)
        pc = PointCloud(len(xyzs))
        pc.mutable_xyzs()[:] = xyzs.T
        meshcat.SetObject(f"{name}/region {i}", pc, point_size=0.025, rgba=colors[i % len(colors)])
    
meshcat = StartMeshcat()


INFO:drake:Meshcat listening for connections at http://localhost:7000


In [60]:
meshcat.DeleteAddedControls()
vv = ModelVisualizer(browser_new=True)
vv.parser().package_map().Add("ciris_pgd", os.path.abspath(''))
# vv.AddModels("/home/sgrg/rlg/SUPERUROP/ciris/models/iiwa14_primitive_collision.urdf")
vv.AddModels("/home/sgrg/rlg/SUPERUROP/ciris/models/clutter_ciris_scenario.dmd.yaml")
vv.Run()

INFO:drake:Meshcat listening for connections at http://localhost:7004
INFO:drake:Click 'Stop Running' or press Esc to quit


In [55]:
#construct our robot
builder = RobotDiagramBuilder()
builder.parser().package_map().Add("ciris_pgd", os.path.abspath(''))

directives_file = "/home/sgrg/rlg/SUPERUROP/ciris/models/clutter_ciris_scenario.dmd.yaml"
directives = LoadModelDirectives(directives_file)
ProcessModelDirectives(directives, builder.plant(), builder.parser())

meshcat_visual_params = MeshcatVisualizerParams()
meshcat_visual_params.delete_on_initialization_event = False
meshcat_visual_params.role = Role.kIllustration
meshcat_visual_params.prefix = "visual"
meshcat_visual_params.visible_by_default = True
meshcat_visual = MeshcatVisualizer.AddToBuilder(
    builder.builder(), builder.scene_graph(), meshcat, meshcat_visual_params)
meshcat_collision_params = MeshcatVisualizerParams()
meshcat_collision_params.delete_on_initialization_event = False
meshcat_collision_params.role = Role.kProximity
meshcat_collision_params.prefix = "collision"
meshcat_collision_params.visible_by_default = False
meshcat_collision = MeshcatVisualizer.AddToBuilder(
    builder.builder(), builder.scene_graph(), meshcat, meshcat_collision_params)

diagram = builder.Build()
simulator = Simulator(diagram)
context = simulator.get_context()
diagram.ForcedPublish(context)

In [42]:
plant = diagram.plant()
for obj in plant.GetFloatingBaseBodies():
    body_index = obj # we currently only have one floating base body
context = simulator.get_context()
plant_context = plant.GetMyContextFromRoot(context)
tf = RigidTransform(
            RotationMatrix(),
            [0.45, 0.67, 0.6],
        )
plant.SetFreeBodyPose(plant_context, plant.get_body(body_index), tf)
diagram.ForcedPublish(context)

In [43]:
# q0 = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
# plant.SetPositions(plant_context, q0)

In [57]:
simulator.AdvanceTo(2.0)

In [51]:
tf = plant.GetFreeBodyPose(plant_context, plant.get_body(body_index))
tf.translation()

array([0.44996586, 0.67000133, 0.58414171])

In [17]:
Ratfk = RationalForwardKinematics(plant)
q_star = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])
q_low = np.array([-2.967060,-2.094395,-2.967060,-2.094395,-2.967060,-2.094395,-3.054326])
tc_low = Ratfk.ComputeSValue(q_low, q_star)
q_high = np.array([2.967060,2.094395,2.967060,2.094395,2.967060,2.094395,3.054326])
tc_high = Ratfk.ComputeSValue(q_high, q_star)
joint_limits = HPolyhedron.MakeBox(tc_low, tc_high)

In [7]:
model = diagram
robot_model_instances = [diagram.plant().GetModelInstanceByName("iiwa")]
edge_step_size = 0.01
collision_checker = SceneGraphCollisionChecker(model=model, robot_model_instances=robot_model_instances, edge_step_size=edge_step_size)

INFO:drake:Allocating contexts to support implicit context parallelism 20


In [ ]:
# def grow_region(start_polytope):
start_polytope = AffineBall.MinimumVolumeCircumscribedEllipsoid([q0])
options = IrisZoOptions.CreateWithArctangentParametrization()
IrisZo(collision_checker, start_polytope, joint_limits, options)

In [7]:
do_viz = True

# The object we will use to perform our certification.
cspace_free_polytope = CspaceFreePolytope(plant, scene_graph, SeparatingPlaneOrder.kAffine, q_star)

## MODEL Vis